# Lab: Multiple Variable Linear Regression in Economics
### Modeling Household Consumption Expenditure

In this lab you will extend single-variable linear regression to **multiple explanatory variables**, using a classic economics application: estimating a household's **consumption function** — how much a household spends each month as a function of its income and demographic characteristics.

This lab mirrors the structure of the general "Multiple Variable Linear Regression" lab, but every example, variable, and interpretation is drawn from **economics** rather than housing prices.

# Outline
- [ 1.1 Goals](#toc_1.1)
- [ 1.2 Tools](#toc_1.2)
- [ 1.3 Notation](#toc_1.3)
- [2 Problem Statement: The Consumption Function](#toc_2)
- [ 2.1 Matrix X containing our examples](#toc_2.1)
- [ 2.2 Parameter vector w, b](#toc_2.2)
- [3 Model Prediction With Multiple Variables](#toc_3)
- [ 3.1 Single Prediction, element by element](#toc_3.1)
- [ 3.2 Single Prediction, vectorized](#toc_3.2)
- [4 Compute Cost With Multiple Variables](#toc_4)
- [5 Gradient Descent With Multiple Variables](#toc_5)
- [ 5.1 Compute Gradient with Multiple Variables](#toc_5.1)
- [ 5.2 Gradient Descent With Multiple Variables](#toc_5.2)
- [6 Economic Interpretation of the Coefficients](#toc_6)
- [7 Why Feature Scale Matters (a preview)](#toc_7)
- [8 Congratulations](#toc_8)


<a name="toc_1.1"></a>
## 1.1 Goals
- Extend the simple regression model to support **multiple economic features** (independent variables)
- Rewrite prediction, cost, and gradient routines to handle a feature vector instead of a single scalar
- Use NumPy's `np.dot` to vectorize the implementation for speed and clarity
- Practice **interpreting regression coefficients economically** (marginal propensity to consume, elasticities, etc.)


<a name="toc_1.2"></a>
## 1.2 Tools
In this lab we will use:
- **NumPy** — the standard library for numerical / vectorized computation in Python
- **Matplotlib** — for plotting cost curves and diagnostic charts


In [ ]:
import copy, math
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=2)  # reduced display precision on numpy arrays
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

<a name="toc_1.3"></a>
## 1.3 Notation

| General <br> Notation | Description | Python |
|:---|:---|:---|
| $a$ | scalar, non-bold | |
| $\mathbf{a}$ | vector, bold | |
| $\mathbf{A}$ | matrix, bold capital | |
| **Regression** | | |
| $\mathbf{X}$ | training example matrix (households × features) | `X_train` |
| $\mathbf{y}$ | training example targets (consumption expenditure) | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | the $i^{th}$ household's feature vector and target | `X[i]`, `y[i]` |
| m | number of households (training examples) | `m` |
| n | number of economic features per household | `n` |
| $\mathbf{w}$ | parameter vector: marginal effects of each feature | `w` |
| $b$ | parameter: intercept / autonomous consumption | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | model prediction: $\mathbf{w}\cdot\mathbf{x}^{(i)}+b$ | `f_wb` |


<a name="toc_2"></a>
# 2 Problem Statement: The Consumption Function

Economists have long modeled household spending using a **consumption function**:

$$C = f(\text{Income}, \text{Household characteristics}) $$

The simplest version — Keynes's consumption function — relates consumption only to income: $C = a + b \cdot \text{Income}$. In this lab we **extend it to multiple explanatory variables**, since real household spending depends on more than income alone.

You are given cross-sectional survey data on **3 households**, with **4 explanatory (feature) variables** and 1 target variable:

| Monthly Disposable Income ($) | Household Size (persons) | Head's Education (years) | Head's Age (years) | Monthly Consumption Expenditure ($) |
|---|---|---|---|---|
| 4360 | 5 | 16 | 45 | 3210 |
| 2950 | 3 | 12 | 40 | 2100 |
| 1560 | 2 | 9  | 35 | 1250 |

Notice that, just like the housing lab this is based on, the features live on **very different numeric scales** — income is in the thousands of dollars while household size is a single digit. This is a realistic feature of economic data (income, prices, and quantities are rarely measured on comparable scales) and will matter later.

Your goal: build a linear regression model that predicts a household's **monthly consumption expenditure** from its income, size, education, and age — then use it to predict spending for a new household, e.g. one earning \$2,000/month, with 3 members, a head with 12 years of education, aged 38.

Run the cell below to create `X_train` and `y_train`.


In [ ]:
X_train = np.array([[4360, 5, 16, 45],
                     [2950, 3, 12, 40],
                     [1560, 2,  9, 35]])
y_train = np.array([3210, 2100, 1250])
feature_names = ['Monthly Income ($)', 'Household Size', "Head's Education (yrs)", "Head's Age (yrs)"]

<a name="toc_2.1"></a>
## 2.1 Matrix X containing our examples

Each row of `X_train` is one household (one training example). With $m$ households (3 here) and $n$ features (4 here), $\mathbf{X}$ is an $(m,n)$ matrix:

$$\mathbf{X} =
\begin{pmatrix}
 x^{(0)}_0 & x^{(0)}_1 & \cdots & x^{(0)}_{n-1} \\
 x^{(1)}_0 & x^{(1)}_1 & \cdots & x^{(1)}_{n-1} \\
 \cdots \\
 x^{(m-1)}_0 & x^{(m-1)}_1 & \cdots & x^{(m-1)}_{n-1}
\end{pmatrix}
$$

- $\mathbf{x}^{(i)}$ is the feature vector for household $i$: (Income, Size, Education, Age)
- $x^{(i)}_j$ is feature $j$ (e.g. Income) for household $i$

Display the input data:


In [ ]:
print(f"X Shape: {X_train.shape}, X Type: {type(X_train)}")
print(X_train)
print(f"y Shape: {y_train.shape}, y Type: {type(y_train)}")
print(y_train)

<a name="toc_2.2"></a>
## 2.2 Parameter vector w, b

- $\mathbf{w}$ has one entry per feature ($n=4$ here): the **marginal effect of each variable on consumption**, holding the others fixed.
  - $w_0$: marginal propensity to consume out of income (extra \$ spent per extra \$ earned)
  - $w_1$: extra spending per additional household member
  - $w_2$: extra spending per extra year of the head's education
  - $w_3$: extra spending per extra year of the head's age
- $b$ is a scalar: roughly, **autonomous consumption** — spending that doesn't depend on the observed features.

$$\mathbf{w} = \begin{pmatrix} w_0 \\ w_1 \\ \cdots \\ w_{n-1} \end{pmatrix}$$

For demonstration, `w_init` and `b_init` below are values close to the least-squares optimum for this tiny dataset (you will recover values like these yourself using gradient descent in Section 5).


In [ ]:
b_init = -0.27722267173752363
w_init = np.array([0.31344815, 129.41729711, 124.31373982, -17.61028998])
print(f"w_init shape: {w_init.shape}, b_init type: {type(b_init)}")

<a name="toc_3"></a>
# 3 Model Prediction With Multiple Variables

The model's prediction with multiple variables is:

$$ f_{\mathbf{w},b}(\mathbf{x}) = w_0x_0 + w_1x_1 + \cdots + w_{n-1}x_{n-1} + b \tag{1}$$

or, in vector notation:

$$ f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w}\cdot\mathbf{x} + b \tag{2}$$

where $\cdot$ is the vector dot product. In economic terms: predicted consumption is a weighted sum of income, household size, education, and age, plus a baseline.


<a name="toc_3.1"></a>
## 3.1 Single Prediction, element by element

Implement equation (1) directly with a loop over each feature.


In [ ]:
def predict_single_loop(x, w, b):
    """
    single predict using linear regression

    Args:
      x (ndarray): Shape (n,) example with multiple features
      w (ndarray): Shape (n,) model parameters
      b (scalar):  model parameter

    Returns:
      p (scalar):  prediction
    """
    n = x.shape[0]
    p = 0
    for i in range(n):
        p_i = x[i] * w[i]
        p = p + p_i
    p = p + b
    return p

In [ ]:
# get a row from our training data (household 0)
x_vec = X_train[0, :]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# make a prediction
f_wb = predict_single_loop(x_vec, w_init, b_init)
print(f"f_wb shape {np.shape(f_wb)}, prediction: {f_wb}")

**Expected Result**:
```
x_vec shape (4,), x_vec value: [4360    5   16   45]
f_wb shape (), prediction: 3209.999984898262
```
`x_vec` is a 1-D NumPy vector with 4 elements — (Income, Size, Education, Age) for household 0. `f_wb` is a scalar prediction of that household's consumption (\$3210, matching the actual target closely, since `w_init`/`b_init` are near-optimal for this data).


<a name="toc_3.2"></a>
## 3.2 Single Prediction, vectorized

Equation (1) is exactly a dot product, equation (2). Use `np.dot()` to vectorize the prediction — this is both shorter and (for larger $n$) much faster than the Python loop.


In [ ]:
def predict(x, w, b):
    """
    single predict using linear regression
    Args:
      x (ndarray): Shape (n,) example with multiple features
      w (ndarray): Shape (n,) model parameters
      b (scalar):             model parameter

    Returns:
      p (scalar):  prediction
    """
    p = np.dot(x, w) + b
    return p

In [ ]:
# get a row from our training data
x_vec = X_train[0, :]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# make a prediction
f_wb = predict(x_vec, w_init, b_init)
print(f"f_wb shape {np.shape(f_wb)}, prediction: {f_wb}")

**Expected Result**:
```
x_vec shape (4,), x_vec value: [4360    5   16   45]
f_wb shape (), prediction: 3209.999984898262
```
Same result as the loop version, but as a single line. Going forward we will use `np.dot` directly inside the cost and gradient routines rather than calling a separate `predict` function.


<a name="toc_4"></a>
# 4 Compute Cost With Multiple Variables

The cost function $J(\mathbf{w},b)$ with multiple variables is:

$$J(\mathbf{w},b) = \frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)^2 \tag{3}$$

where

$$f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w}\cdot\mathbf{x}^{(i)} + b \tag{4}$$

Unlike the single-feature case, $\mathbf{w}$ and $\mathbf{x}^{(i)}$ are now vectors. This is simply the (mean-halved) sum of squared errors between predicted and actual household consumption, exactly as before — just with a vector dot product inside.


In [ ]:
def compute_cost(X, y, w, b):
    """
    compute cost
    Args:
      X (ndarray (m,n)): Data, m households with n economic features
      y (ndarray (m,)) : target values (consumption expenditure)
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        f_wb_i = np.dot(X[i], w) + b        # scalar prediction for household i
        cost = cost + (f_wb_i - y[i]) ** 2  # squared error
    cost = cost / (2 * m)
    return cost

In [ ]:
# Compute and display cost using our pre-chosen near-optimal parameters
cost = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at optimal w : {cost}')

**Expected Result**: Cost at optimal w : `6.01655853824931e-11` (essentially zero, confirming `w_init`/`b_init` fit these 3 households almost exactly).


<a name="toc_5"></a>
# 5 Gradient Descent With Multiple Variables

Gradient descent for multiple variables:

$$\begin{align*}
\text{repeat until convergence:}\ \{ \\
& w_j = w_j - \alpha\frac{\partial J(\mathbf{w},b)}{\partial w_j} \quad \text{for } j=0..n-1 \tag{5}\\
& b = b - \alpha\frac{\partial J(\mathbf{w},b)}{\partial b} \\
\}
\end{align*}$$

where $w_j$ and $b$ are updated **simultaneously**, and

$$\frac{\partial J(\mathbf{w},b)}{\partial w_j} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)x_j^{(i)} \tag{6}$$

$$\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right) \tag{7}$$

- $m$ is the number of households in the dataset
- $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the model's predicted spending; $y^{(i)}$ is actual spending


<a name="toc_5.1"></a>
## 5.1 Compute Gradient with Multiple Variables

One implementation pattern:
- outer loop over the $m$ households: accumulate $\partial J/\partial b$ directly
- inner loop over the $n$ features: accumulate $\partial J/\partial w_j$ for each $j$


In [ ]:
def compute_gradient(X, y, w, b):
    """
    Computes the gradient for linear regression
    Args:
      X (ndarray (m,n)): Data, m households with n economic features
      y (ndarray (m,)) : target values (consumption expenditure)
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w.
      dj_db (scalar):       The gradient of the cost w.r.t. the parameter b.
    """
    m, n = X.shape
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):
        err = (np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err * X[i, j]
        dj_db = dj_db + err
    dj_dw = dj_dw / m
    dj_db = dj_db / m

    return dj_db, dj_dw

In [ ]:
# Compute and display gradient
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
print(f'dj_db at initial w,b: {tmp_dj_db}')
print(f'dj_dw at initial w,b: \n {tmp_dj_dw}')

**Expected Result**:
```
dj_db at initial w,b: -1.0225071188566895e-05
dj_dw at initial w,b:
 [-3.48e-02 -3.90e-05 -1.37e-04 -4.25e-04]
```


<a name="toc_5.2"></a>
## 5.2 Gradient Descent With Multiple Variables

The routine below implements equation (5) above by repeatedly calling your `compute_gradient` and `compute_cost` functions. This function is provided for you (it doesn't change when moving from 1 to $n$ features).


In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent to learn w and b. Updates w and b by taking
    num_iters gradient steps with learning rate alpha

    Args:
      X (ndarray (m,n))   : Data, m households with n economic features
      y (ndarray (m,))    : target values (consumption expenditure)
      w_in (ndarray (n,)) : initial model parameters
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent

    Returns:
      w (ndarray (n,)) : Updated values of parameters
      b (scalar)       : Updated value of parameter
      J_history (list) : cost at each iteration (for plotting)
    """
    J_history = []
    w = copy.deepcopy(w_in)
    b = b_in

    for i in range(num_iters):
        dj_db, dj_dw = gradient_function(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        if i < 100000:  # prevent resource exhaustion
            J_history.append(cost_function(X, y, w, b))

        if i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}")

    return w, b, J_history

Now run gradient descent starting from $\mathbf{w}=\mathbf{0}$, $b=0$, and see what parameters it recovers.


In [ ]:
# initialize parameters
initial_w = np.zeros_like(w_init)
initial_b = 0.
# gradient descent settings
iterations = 1000
alpha = 5.0e-8
# run gradient descent
w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b,
                                             compute_cost, compute_gradient,
                                             alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
m, _ = X_train.shape
for i in range(m):
    print(f"prediction: {np.dot(X_train[i], w_final) + b_final:0.2f}, target value: {y_train[i]}")

**Expected Result**:
```
b,w found by gradient descent: 0.00,[0.73 0.   0.01 0.03]
prediction: 3201.95, target value: 3210
prediction: 2166.76, target value: 2100
prediction: 1146.26, target value: 1250
```


In [ ]:
# plot cost versus iteration
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist)
ax2.plot(100 + np.arange(len(J_hist[100:])), J_hist[100:])
ax1.set_title("Cost vs. iteration");  ax2.set_title("Cost vs. iteration (tail)")
ax1.set_ylabel('Cost');               ax2.set_ylabel('Cost')
ax1.set_xlabel('iteration step');     ax2.set_xlabel('iteration step')
plt.show()

*These results are not very inspiring* — the cost is still declining after 1000 iterations, and predictions are noticeably off for some households. Note that we had to use a **tiny learning rate** ($\alpha=5\times10^{-8}$) just to keep gradient descent from diverging, because `Income` (thousands) and `Household Size` (single digits) sit on wildly different numeric scales. Larger, more natural learning rates blow up (try `alpha = 5e-7` and watch it diverge to `nan`!). **Feature scaling** — the subject of the next lab — fixes this.


<a name="toc_6"></a>
# 6 Economic Interpretation of the Coefficients

One of the biggest differences between a "generic" machine learning exercise and applied econometrics is that **we care about what the coefficients mean**, not just predictive accuracy. Recall equation (2):

$$ \hat{C} = w_0\cdot\text{Income} + w_1\cdot\text{Size} + w_2\cdot\text{Education} + w_3\cdot\text{Age} + b $$

- **$w_0$ — Marginal Propensity to Consume (MPC):** the extra dollars spent for every extra dollar of income, holding size/education/age fixed. Keynesian theory predicts $0 < w_0 < 1$.
- **$w_1$:** the extra monthly spending associated with one additional household member (more mouths to feed), holding income fixed.
- **$w_2$:** the association between the household head's education and spending, holding income fixed — often interpreted as a proxy for permanent income, financial literacy, or consumption preferences not captured by current income.
- **$w_3$:** the association between the head's age and spending, holding the others fixed — related to *life-cycle* consumption theory (spending patterns differ across a person's life).
- **$b$:** predicted consumption when all features are zero — rarely meaningful on its own for small samples, but mathematically it is the model's intercept.

Print the near-optimal coefficients estimated earlier with their sign and rough interpretation:


In [ ]:
for name, coef in zip(feature_names, w_init):
    direction = "increases" if coef > 0 else "decreases"
    print(f"{name:28s}: w = {coef:9.4f}  -> spending {direction} by ${abs(coef):.2f} per unit increase")
print(f"{'Intercept (b)':28s}: b = {b_init:9.4f}")

> **Caution — correlation vs. causation.** Even though $w_0\approx0.31$ *looks like* a textbook MPC, these coefficients come from only 3 observations with no controls for confounders (e.g., wealth, region, family composition beyond size). In real applied economics, causal claims about the MPC require much larger samples, robust standard errors, and often quasi-experimental designs (e.g., using tax-rebate "natural experiments"). Treat this lab as a **numerical/statistical exercise**, not a causal study.


<a name="toc_7"></a>
# 7 Why Feature Scale Matters (a preview)

Compare the *ranges* of the four features:


In [ ]:
for name, col in zip(feature_names, X_train.T):
    print(f"{name:28s}: min={col.min():8.1f}  max={col.max():8.1f}  range={col.max()-col.min():8.1f}")

Income spans thousands of dollars while household size spans a handful of people. Because gradient descent takes the *same* step size $\alpha$ for every parameter, a step large enough to make progress on the small-range features (size, education, age) is wildly too large for the big-range feature (income), and vice-versa — this is exactly why we needed such a tiny `alpha` above, and why convergence was so slow. The next lab in this series (**Feature Scaling & Learning Rate**) shows how z-score normalization fixes this and lets gradient descent converge quickly with a much larger learning rate.


<a name="toc_8"></a>
# 8 Congratulations!

In this lab you:
- Extended linear regression routines to support **multiple economic features**
- Implemented `predict`, `compute_cost`, and `compute_gradient` for the multivariate case
- Used `np.dot` to vectorize the implementation
- Ran batch gradient descent to estimate a simple **consumption function**
- Practiced interpreting regression coefficients in economic terms (MPC, life-cycle effects)
- Saw first-hand why **feature scaling** will matter in the next lab
